# MLB Data Exploration Notebook

This notebook explores the MLB 2026 season data pipeline.

## Overview
- Extract data from MLB Stats API
- Explore batting and pitching statistics
- Calculate sabermetric features
- Visualize distributions and relationships

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.api.mlb_client import MLBClient
from src.etl.extract_mlb import MLBDataExtractor
from src.utils.config_loader import load_config

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Configuration

In [ ]:
# Load configuration
config = load_config('config/settings.yaml')
features_config = load_config('config/features.yaml')

print('Configuration loaded successfully')
print(f"Season: {config.get('execution', {}).get('seasons', [2026])[0]}")
print(f"Random state: {config.get('model', {}).get('random_state', 42)}")
print(f"Train/Val/Test split: {config.get('model', {}).get('train_ratio')}/{config.get('model', {}).get('val_ratio')}/{config.get('model', {}).get('test_ratio')}")

## 2. Extract Data from MLB API

In [ ]:
# Initialize extractor
extractor = MLBDataExtractor(config)

# Extract batting stats
print('Extracting batting statistics...')
df_batting = extractor.extract_player_stats(stat_type='hitting', season=2026)
print(f'Extracted {len(df_batting)} batting records')
print(f'\nBatting data shape: {df_batting.shape}')
print(f'\nColumns: {df_batting.columns.tolist()}')

In [ ]:
# Display sample batting data
df_batting.head()

In [ ]:
# Extract pitching stats
print('Extracting pitching statistics...')
df_pitching = extractor.extract_player_stats(stat_type='pitching', season=2026)
print(f'Extracted {len(df_pitching)} pitching records')
print(f'\nPitching data shape: {df_pitching.shape}')

In [ ]:
# Extract standings
print('Extracting standings...')
df_standings = extractor.extract_standings(season=2026)
print(f'Extracted {len(df_standings)} standings records')
df_standings.head()

## 3. Data Exploration

In [ ]:
# Batting statistics summary
print('Batting Statistics Summary:')
print(f'Total players: {df_batting["player_id"].nunique()}')
print(f'Total records: {len(df_batting)}')
print(f'\nMissing values by column:')
print(df_batting.isnull().sum())

In [ ]:
# Pitching statistics summary
print('Pitching Statistics Summary:')
print(f'Total pitchers: {df_pitching["player_id"].nunique()}')
print(f'Total records: {len(df_pitching)}')
print(f'\nPitchers with games pitched:')
if 'games' in df_pitching.columns:
    print(df_pitching['games'].describe())

## 4. Sabermetric Calculations

In [ ]:
# Calculate batting features (example)
if 'hits' in df_batting.columns and 'at_bats' in df_batting.columns:
    df_batting['avg'] = df_batting['hits'] / df_batting['at_bats'].replace(0, np.nan)
    print('Batting Average Distribution:')
    print(df_batting['avg'].describe())

In [ ]:
# Calculate pitching features (example)
if 'earned_runs' in df_pitching.columns and 'innings_pitched' in df_pitching.columns:
    df_pitching['era'] = (df_pitching['earned_runs'] * 9) / df_pitching['innings_pitched'].replace(0, np.nan)
    print('ERA Distribution:')
    print(df_pitching['era'].describe())

## 5. Visualizations

In [ ]:
# Batting Average distribution
if 'avg' in df_batting.columns:
    fig, ax = plt.subplots()
    df_batting['avg'].hist(bins=30, ax=ax, edgecolor='black')
    ax.set_xlabel('Batting Average')
    ax.set_ylabel('Frequency')
    ax.set_title('Distribution of Batting Averages (2026)')
    plt.show()

In [ ]:
# ERA distribution
if 'era' in df_pitching.columns:
    fig, ax = plt.subplots()
    df_pitching['era'].dropna().hist(bins=30, ax=ax, edgecolor='black')
    ax.set_xlabel('ERA')
    ax.set_ylabel('Frequency')
    ax.set_title('Distribution of ERA (2026)')
    plt.show()

## 6. Standings Analysis

In [ ]:
# Top teams by wins
if 'wins' in df_standings.columns:
    top_teams = df_standings.nlargest(10, 'wins')[['team_name', 'wins', 'losses', 'win_pct']]
    print('Top 10 Teams by Wins:')
    print(top_teams)

In [ ]:
# Close extractor
extractor.close()
print('Extractor connection closed')

## Summary

This exploration demonstrates:
- Successful data extraction from MLB Stats API
- Calculation of basic sabermetric features
- Data quality assessment
- Basic visualization of player performance distributions

Next steps: Feature engineering pipeline and ML model training